<a href="https://colab.research.google.com/github/sotetsuk/LectureColab/blob/main/reinforce_cartpole.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 応用機械学習 #3：CartPole で REINFORCE を動かす

講義資料の CartPole の実験を、上から順に実行して確かめる。
ランタイムは CPU のままでよい。
「ランタイム」→「すべてのセルを実行」で、全体が15分ほどで終わる。

In [ ]:
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from IPython.display import HTML
from matplotlib import animation
from torch.distributions import Categorical

N_SEEDS = 2    # 比較する乱数シードの数 (講義資料は 5. Colab の CPU では 1 シード 2 分ほどかかる)
UPDATES = 300  # 更新回数

# 図の色と線種 (講義資料と同じ)
COLOR = {"none": "#0072B2", "mc": "#D55E00", "td": "#009E73"}
LS = {"none": "-", "mc": "--", "td": ":"}
LABEL = {"none": "no baseline", "mc": "value baseline (MC)", "td": "value baseline (TD)"}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "animation.embed_limit": 100})

## 1. CartPole 環境

- 状態は $s=(x,\dot x,\vartheta,\dot\vartheta)$（台車の位置と速度、棒の角度と角速度）、行動は左に押す（0）か右に押す（1）
- 棒が立っている間、毎ステップ報酬 $+1$ が得られる
- 棒が12度より傾くか、台車が端を越えると終了する。500ステップで打ち切る

次のセルが環境の実装である（Gym の `CartPole-v1` と同じ運動方程式）。
$n$ 個の環境をまとめて進める。

In [ ]:
# @title CartPole 環境
class CartPole:
    """n 個の CartPole を同時に進める. 状態は (x, x', θ, θ'), 行動は {0: 左, 1: 右}."""

    def __init__(self, n, device="cpu", max_steps=500):
        self.n, self.device, self.max_steps = n, device, max_steps
        self.gravity, self.mass_cart, self.mass_pole = 9.8, 1.0, 0.1
        self.length, self.force_mag, self.tau = 0.5, 10.0, 0.02
        self.total_mass = self.mass_cart + self.mass_pole
        self.pole_mass_length = self.mass_pole * self.length
        self.x_threshold = 2.4
        self.theta_threshold = 12 * 2 * math.pi / 360

    def reset(self):
        self.state = (torch.rand(self.n, 4, device=self.device) - 0.5) * 0.1
        return self.state.clone()

    def step(self, action):
        x, x_dot, theta, theta_dot = self.state.unbind(1)
        force = torch.where(action == 1, self.force_mag, -self.force_mag)
        cos_t, sin_t = torch.cos(theta), torch.sin(theta)
        temp = (force + self.pole_mass_length * theta_dot**2 * sin_t) / self.total_mass
        theta_acc = (self.gravity * sin_t - cos_t * temp) / (
            self.length * (4.0 / 3.0 - self.mass_pole * cos_t**2 / self.total_mass)
        )
        x_acc = temp - self.pole_mass_length * theta_acc * cos_t / self.total_mass
        x = x + self.tau * x_dot
        x_dot = x_dot + self.tau * x_acc
        theta = theta + self.tau * theta_dot
        theta_dot = theta_dot + self.tau * theta_acc
        self.state = torch.stack([x, x_dot, theta, theta_dot], 1)
        terminated = (x.abs() > self.x_threshold) | (theta.abs() > self.theta_threshold)
        reward = torch.ones(self.n, device=self.device)  # 立っている限り毎ステップ +1
        return self.state.clone(), reward, terminated

In [ ]:
env = CartPole(n=3)                                        # 3 個の環境をまとめて扱う
s = env.reset()                                            # 初期状態 [3, 4]
s_next, r, terminated = env.step(torch.tensor([0, 1, 1]))  # 環境ごとの行動を与えて 1 ステップ進める
print(s_next, r, terminated, sep="\n")

In [ ]:
# @title 可視化用の関数 rollout, animate
@torch.no_grad()
def rollout(act, max_steps=500):
    """1 個の環境を act(s) で動かし, 状態 [T+1, 4] と行動 [T] を返す.

    終了したときは, そのあと押さずに棒が倒れきるまで進めた状態を, 表示用に states の後ろに足す.
    """
    env = CartPole(n=1)
    s = env.reset()
    states, actions = [s[0]], []
    for _ in range(max_steps):
        a = act(s)
        s, r, terminated = env.step(a)
        states.append(s[0])
        actions.append(a[0])
        if terminated.item():
            env.force_mag = 0.0  # 終了後は押さない
            while abs(s[0, 2]) < math.pi / 2 and len(states) < len(actions) + 60:
                s, _, _ = env.step(a)
                states.append(s[0])
            break
    return torch.stack(states).numpy(), torch.stack(actions).numpy()


def animate(states, actions, title="", values=None, every=2, slow=1):
    """rollout の結果をアニメーションにする. every ステップごとに 1 コマ描き, slow 倍に遅くして再生する.

    矢印は押した向き, 点線は終了となる傾き (12 度), 灰色の棒は終了後. values を渡すと V(s_t) も表示する.
    """
    T = len(actions)
    ended = len(states) > T + 1  # 打ち切りではなく終了したか
    pole_len, cart_w, cart_h = 1.0, 0.5, 0.3  # 棒の長さは 2 × 0.5
    lim = 12 * math.pi / 180
    fig, ax = plt.subplots(figsize=(6, 2.3), dpi=80)
    ax.set_xlim(-2.9, 2.9)
    ax.set_ylim(-0.2, 1.4)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(title, fontsize=11)
    ax.plot([-2.9, 2.9], [0, 0], color="0.5", lw=1)
    for x in (-2.4, 2.4):  # 台車がこの外に出ると終了
        ax.plot([x, x], [-0.15, 0.7], color="0.6", ls="--", lw=1)
    cart = plt.Rectangle((0, 0), cart_w, cart_h, color="0.3")
    ax.add_patch(cart)
    guides = [ax.plot([], [], color="0.6", ls=":", lw=1)[0] for _ in range(2)]
    (pole,) = ax.plot([], [], lw=6, solid_capstyle="round")
    arrow = ax.annotate("", xy=(0, 0), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="#0072B2", lw=2))
    text = ax.text(-2.85, 1.35, "", fontsize=11, va="top", family="monospace")

    def draw(t):
        x, _, th, _ = states[t]
        th = np.clip(th, -np.pi / 2, np.pi / 2)  # 横倒しより先は描かない
        cart.set_xy((x - cart_w / 2, 0))
        for g, a in zip(guides, (-lim, lim)):
            g.set_data([x, x + pole_len * np.sin(a)], [cart_h, cart_h + pole_len * np.cos(a)])
        pole.set_data([x, x + pole_len * np.sin(th)], [cart_h, cart_h + pole_len * np.cos(th)])
        pole.set_color("#E69F00" if t <= T else "0.7")
        if t < T:
            d = 1 if actions[t] == 1 else -1
            arrow.set_position((x + d * cart_w / 2, cart_h / 2))  # 矢印の根元
            arrow.xy = (x + d * (cart_w / 2 + 0.4), cart_h / 2)   # 矢印の先
        arrow.set_visible(t < T)
        label = f"t = {min(t, T):3d}"
        if values is not None and t < len(values):
            label += f"   V(s) = {values[t]:5.1f}"
        if ended and t >= T:
            label += "   terminated"
        text.set_text(label)
        return cart, pole, arrow, text, *guides

    frames = list(range(0, len(states), every))
    if frames[-1] != len(states) - 1:
        frames.append(len(states) - 1)
    anim = animation.FuncAnimation(fig, draw, frames=frames, interval=20 * every * slow, repeat=False)  # slow=1 で実時間
    plt.close(fig)
    try:
        return HTML(anim.to_html5_video())  # 動画にする (ffmpeg が必要. Colab には入っている)
    except RuntimeError:  # ffmpeg が無い環境では, コマを並べた HTML にする
        return HTML(anim.to_jshtml(default_mode="once"))

ランダムに行動を選んで1エピソード動かす（4分の1の速さで再生）。
棒が点線（12度）を越えたところで終了する。
終了後は台車を押さずに、棒が倒れきるまでを灰色で表示する。

In [ ]:
torch.manual_seed(2)
random_act = lambda s: torch.randint(0, 2, (s.shape[0],))
states, actions = rollout(random_act)
print(f"{len(actions)} ステップで終了")
animate(states, actions, title="random policy", every=1, slow=4)

## 2. REINFORCE

### 方策 NN

状態を受け取り、2つの行動のロジットを返す。

In [ ]:
def mlp(n_in, n_out, hidden=64):
    return nn.Sequential(
        nn.Linear(n_in, hidden), nn.Tanh(),
        nn.Linear(hidden, hidden), nn.Tanh(),
        nn.Linear(hidden, n_out))


torch.manual_seed(0)
policy = mlp(4, 2)
s = CartPole(n=3).reset()
dist = Categorical(logits=policy(s))
print("π(a|s) =", dist.probs)  # 学習前はほぼ 0.5 ずつ
print("サンプルした行動:", dist.sample())

### データ収集

$M$ 本のエピソードを並列に、すべて終わるまで進める。
`ALIVE[t, i]` は、エピソード $i$ が時刻 $t$ にまだ続いているかを表す。
終了後の時刻は損失から除く。
500ステップの打ち切りは終了ではないので、`ALIVE` を変えない。

In [ ]:
@torch.no_grad()
def collect_episodes(env, policy):
    """返り値はどれも [T, M] (状態は [T, M, 4]). S_next と TERM は第 4 節の TD で使う."""
    s = env.reset()
    alive = torch.ones(env.n, dtype=torch.bool)
    S, A, R, S_next, ALIVE, TERM = [], [], [], [], [], []
    for _ in range(env.max_steps):                  # 500 ステップで打ち切る
        a = Categorical(logits=policy(s)).sample()  # 方策から行動をサンプル
        s_next, r, terminated = env.step(a)
        S.append(s), A.append(a), R.append(r), S_next.append(s_next)
        ALIVE.append(alive), TERM.append(terminated)
        alive = alive & ~terminated
        s = s_next
        if not alive.any():
            break
    return [torch.stack(x) for x in (S, A, R, S_next, ALIVE, TERM)]


# 学習前の方策で 16 本集め, ALIVE を色で示す (色の付いた部分がエピソードの続いている時刻)
torch.manual_seed(0)
S, A, R, S_next, ALIVE, TERM = collect_episodes(CartPole(n=16), policy)
print("S:", tuple(S.shape), " A:", tuple(A.shape), " ALIVE:", tuple(ALIVE.shape))
fig, ax = plt.subplots(figsize=(6, 2.2))
ax.imshow(ALIVE.T.float(), aspect="auto", cmap="Blues", vmin=0, vmax=1.3, interpolation="nearest")
ax.set_xlabel("time step $t$")
ax.set_ylabel("episode $i$")
plt.show()

### 収益

$G_t=r_{t+1}+\gamma G_{t+1}$ を最後の時刻から順に計算する。

In [ ]:
def discounted_returns(R, ALIVE, gamma):
    G = torch.zeros_like(R)
    running = torch.zeros(R.shape[1])
    for t in reversed(range(R.shape[0])):
        running = (R[t] + gamma * running) * ALIVE[t]  # 終了後は 0
        G[t] = running
    return G


G = discounted_returns(R, ALIVE, gamma=0.99)
fig, ax = plt.subplots(figsize=(4.5, 2.6))
for i in range(4):
    T_i = int(ALIVE[:, i].sum())
    ax.plot(range(T_i), G[:T_i, i], marker=".", ms=4, label=f"episode {i} ($T_i$ = {T_i})")
ax.set_xlabel("time step $t$")
ax.set_ylabel("$G_t$")
ax.legend(frameon=False)
plt.show()

$\gamma=0.99$ なので、500ステップ立ち続けても $G_0\approx99.3$ である。
以下の学習曲線の「収益」は割引なしの和（最大500）を表す。

### 損失と1回の更新

$$
\widehat{\mathcal L}_{\mathrm{RF}}(\theta)
=-\frac{1}{M}\sum_{i=1}^M\sum_{t=0}^{T_i-1}
\underbrace{\gamma^t\bigl(G_t^{(i)}-b(s_t^{(i)})\bigr)}_{W_t^{(i)}}
\log\pi_\theta(a_t^{(i)}|s_t^{(i)})
$$

まずはベースラインなし（$b=0$）で動かす。

In [ ]:
torch.manual_seed(0)
policy = mlp(4, 2)
opt = torch.optim.Adam(policy.parameters(), lr=1e-3)
gamma = 0.99

S, A, R, S_next, ALIVE, TERM = collect_episodes(CartPole(n=16), policy)
G = discounted_returns(R, ALIVE, gamma)
discount_t = gamma ** torch.arange(len(R)).unsqueeze(1)  # γ^t, 形は [T, 1]

baseline = 0                                       # ベースラインなし (b = 0)
W = discount_t * (G - baseline)                    # 重み
logp = Categorical(logits=policy(S)).log_prob(A)   # log π_θ(a_t | s_t)
loss = -(W.detach() * logp * ALIVE).sum(0).mean()  # 重みは定数扱い. 時刻で和, エピソードで平均
opt.zero_grad(); loss.backward(); opt.step()
print("loss =", loss.item())

### 学習

上の更新を、データを集め直しながら繰り返す。
10更新ごとに、勾配のばらつき $\frac1M\sum_i\|g_i-\bar g\|^2$（$g_i$ はエピソード $i$ だけから計算した勾配）を記録する。

In [ ]:
# @title 勾配のばらつきを測る関数 grad_variance
def grad_variance(policy, S, A, W, ALIVE):
    """各エピソード i だけから計算した勾配 g_i の, 平均 ḡ まわりのばらつき (1/M) Σ_i ||g_i - ḡ||²."""
    logp = Categorical(logits=policy(S)).log_prob(A)
    losses = -(W.detach() * logp * ALIVE).sum(0)  # エピソードごとの損失 [M]
    params = list(policy.parameters())
    grads = []
    for loss_i in losses:
        g = torch.autograd.grad(loss_i, params, retain_graph=True)
        grads.append(torch.cat([x.reshape(-1) for x in g]))
    g = torch.stack(grads)  # [M, パラメータ数]
    return (g - g.mean(0)).pow(2).sum(1).mean().item()

In [ ]:
def train_reinforce(seed, M=16, updates=UPDATES, gamma=0.99, lr=1e-3):
    torch.manual_seed(seed)
    env = CartPole(n=M)
    policy = mlp(4, 2)
    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    log = {"env_steps": [], "return": [], "grad_var": []}
    env_steps = 0
    for update in range(1, updates + 1):
        S, A, R, S_next, ALIVE, TERM = collect_episodes(env, policy)
        env_steps += int(ALIVE.sum())                      # 環境とやり取りしたステップ数
        G = discounted_returns(R, ALIVE, gamma)
        discount_t = gamma ** torch.arange(len(R)).unsqueeze(1)

        baseline = 0                                       # ベースラインなし (b = 0)
        W = discount_t * (G - baseline)
        if update % 10 == 0:
            log["grad_var"].append((env_steps, grad_variance(policy, S, A, W, ALIVE)))
        logp = Categorical(logits=policy(S)).log_prob(A)
        loss = -(W.detach() * logp * ALIVE).sum(0).mean()
        opt.zero_grad(); loss.backward(); opt.step()

        log["env_steps"].append(env_steps)
        log["return"].append((R * ALIVE).sum(0).mean().item())  # 割引なしの収益の平均
        if update % 100 == 0:
            print(f"  update {update}  env steps {env_steps:7d}  return {log['return'][-1]:5.1f}")
    return {"policy": policy, "log": log}


def run_seeds(train, **kwargs):
    """シード 0, 1, ..., N_SEEDS-1 で train を実行し, 結果をリストで返す."""
    runs = []
    for seed in range(N_SEEDS):
        t0 = time.time()
        print(f"seed {seed}")
        runs.append(train(seed, **kwargs))
        print(f"  ({time.time() - t0:.0f} 秒)")
    return runs


results = {}  # 手法ごとに, シードごとの学習結果を入れる
results["none"] = run_seeds(train_reinforce)

In [ ]:
# @title 学習曲線を描く関数 plot_curves
YLABEL = {"return": "return", "grad_var": "gradient variance", "critic_rmse": "critic RMSE"}


def series(log, field):
    """log から (環境ステップ数, 値) の列を取り出す."""
    if field == "grad_var":
        x, y = zip(*log["grad_var"])
    else:
        x, y = log["env_steps"], log[field]
    return np.array(x, dtype=float), np.array(y, dtype=float)


def plot_curves(keys, field="return", ax=None, band=True):
    """band=True なら平均 (線) と最小〜最大の範囲 (帯), False なら各シード (細線) と平均 (太線)."""
    if ax is None:
        _, ax = plt.subplots(figsize=(4.5, 2.8))
    for key in keys:
        xs, ys = zip(*(series(run["log"], field) for run in results[key]))
        grid = np.linspace(max(x[0] for x in xs), min(x[-1] for x in xs), 200)
        Y = np.array([np.interp(grid, x, y) for x, y in zip(xs, ys)])  # シード間で横軸を揃える
        if band:
            ax.fill_between(grid, Y.min(0), Y.max(0), color=COLOR[key], alpha=0.2, lw=0)
        else:
            for x, y in zip(xs, ys):
                ax.plot(x, y, color=COLOR[key], alpha=0.4, lw=0.7)
        ax.plot(grid, Y.mean(0), color=COLOR[key], ls=LS[key], lw=2, label=LABEL[key])
    ax.set_xlabel("environment steps")
    ax.set_ylabel(YLABEL[field])
    if field == "return":
        ax.axhline(500, color="0.6", lw=0.8, ls=":")
        ax.set_ylim(0, 540)
    if field == "grad_var":
        ax.set_yscale("log")
    ax.legend(frameon=False, fontsize=9)
    return ax

In [ ]:
plot_curves(["none"], band=False)  # 細線は各シード, 太線はその平均
plt.show()

収益は500に届くが、届いたあとも大きく落ち込むことがあり、シード間の差も大きい。

### 学習した方策

最後の収益が最も高かったシードの方策で、1エピソード動かす。

In [ ]:
best = max(results["none"], key=lambda run: run["log"]["return"][-1])
policy_none = best["policy"]
torch.manual_seed(1)
states, actions = rollout(lambda s: Categorical(logits=policy_none(s)).sample())
print(f"{len(actions)} ステップ")
animate(states, actions, title="REINFORCE (no baseline)")

In [ ]:
# @title 右へ押す確率 (台車が中央で止まっているとき)
def plot_policy_map(policy, ax, title=""):
    th, thd = np.meshgrid(np.linspace(-0.21, 0.21, 101), np.linspace(-1.5, 1.5, 101))
    s = torch.zeros(th.size, 4)  # x = x' = 0
    s[:, 2], s[:, 3] = torch.tensor(th.ravel()), torch.tensor(thd.ravel())
    with torch.no_grad():
        p_right = Categorical(logits=policy(s)).probs[:, 1].reshape(th.shape)
    im = ax.pcolormesh(th, thd, p_right, cmap="RdBu_r", vmin=0, vmax=1, shading="auto")
    ax.set_xlabel(r"pole angle $\vartheta$ [rad]")
    ax.set_ylabel(r"angular velocity $\dot\vartheta$")
    ax.set_title(title, fontsize=10)
    return im


torch.manual_seed(0)
fig, axes = plt.subplots(1, 2, figsize=(8, 2.8), constrained_layout=True)
plot_policy_map(mlp(4, 2), axes[0], "before training")
im = plot_policy_map(policy_none, axes[1], "after training")
fig.colorbar(im, ax=axes, label="P(push right)")
plt.show()

学習後の方策は、主に角速度で押す向きを決め、棒が右へ倒れつつある（$\dot\vartheta>0$）ときに右へ押す。
棒の倒れる側へ台車を動かして支えている。

## 3. 価値ベースライン（MC）

$G_t$ から行動に依存しない $b(s_t)$ を引いても、勾配の期待値は変わらず、ばらつきを減らせる。
そこで $b(s_t)=V_\phi(s_t)$ とし、critic $V_\phi$ を $G_t$ への回帰で学習する。

$$
\mathcal L(\phi)=\frac{1}{N}\sum_{(s,G)}\bigl(V_\phi(s)-G\bigr)^2,
\qquad N=\sum_i T_i
$$

`train_reinforce` との違いは `# ★` の行である。

In [ ]:
def train_with_critic(seed, critic_target="mc", M=16, updates=UPDATES, gamma=0.99, lr=1e-3,
                      lr_critic=3e-3, critic_steps=20):
    torch.manual_seed(seed)
    env = CartPole(n=M)
    policy = mlp(4, 2)
    critic = mlp(4, 1)                                           # ★ V_φ(s)
    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    opt_v = torch.optim.Adam(critic.parameters(), lr=lr_critic)  # ★
    log = {"env_steps": [], "return": [], "grad_var": [], "critic_rmse": []}
    env_steps = 0
    for update in range(1, updates + 1):
        S, A, R, S_next, ALIVE, TERM = collect_episodes(env, policy)
        env_steps += int(ALIVE.sum())
        G = discounted_returns(R, ALIVE, gamma)
        discount_t = gamma ** torch.arange(len(R)).unsqueeze(1)

        baseline = critic(S).squeeze(-1)                         # ★ critic を更新する前の V_φ(s_t)
        W = discount_t * (G - baseline)
        if update % 10 == 0:
            log["grad_var"].append((env_steps, grad_variance(policy, S, A, W, ALIVE)))
        logp = Categorical(logits=policy(S)).log_prob(A)
        loss = -(W.detach() * logp * ALIVE).sum(0).mean()        # detach で critic に勾配は流れない
        opt.zero_grad(); loss.backward(); opt.step()             # actor の更新

        with torch.no_grad():                                    # ★ critic の学習ターゲット (定数)
            if critic_target == "mc":
                target = G                                       # ★ MC ターゲット
            else:
                v_next = critic(S_next).squeeze(-1) * (~TERM)    # ★ TD ターゲット (第 4 節). 終端なら 0
                target = R + gamma * v_next                      # ★
        for _ in range(critic_steps):                            # ★ critic の更新
            v = critic(S).squeeze(-1)                            # ★
            loss_v = ((v - target) ** 2 * ALIVE).sum() / ALIVE.sum()  # ★
            opt_v.zero_grad(); loss_v.backward(); opt_v.step()   # ★

        with torch.no_grad():  # 実際の収益 G_t に対する critic の誤差 (RMSE) を記録する
            v = critic(S).squeeze(-1)
            log["critic_rmse"].append(((v - G) ** 2 * ALIVE).sum().div(ALIVE.sum()).sqrt().item())
        log["env_steps"].append(env_steps)
        log["return"].append((R * ALIVE).sum(0).mean().item())
        if update % 100 == 0:
            print(f"  update {update}  env steps {env_steps:7d}  return {log['return'][-1]:5.1f}"
                  f"  critic RMSE {log['critic_rmse'][-1]:5.1f}")
    return {"policy": policy, "critic": critic, "log": log}


results["mc"] = run_seeds(train_with_critic, critic_target="mc")

In [ ]:
# 線はシードの平均, 帯は最小から最大の範囲
fig, axes = plt.subplots(1, 2, figsize=(9, 2.8), constrained_layout=True)
plot_curves(["none", "mc"], "return", ax=axes[0])
plot_curves(["none", "mc"], "grad_var", ax=axes[1])
plt.show()

価値ベースラインを入れると、500に届いたあとの落ち込みが小さくなり、勾配のばらつき（右、縦軸は対数）も小さくなる。

### critic の予測

左は学習中の critic の誤差、右は学習後の方策で集め直した16エピソードでの予測と実際の収益である（シード0）。

In [ ]:
run = results["mc"][0]
torch.manual_seed(100)
S, A, R, S_next, ALIVE, TERM = collect_episodes(CartPole(n=16), run["policy"])
G = discounted_returns(R, ALIVE, 0.99)
with torch.no_grad():
    v = run["critic"](S).squeeze(-1)
G_alive, v_alive = G[ALIVE], v[ALIVE]
idx = torch.randperm(len(G_alive))[:1000]  # 1000 点だけ描く

fig, axes = plt.subplots(1, 2, figsize=(9, 2.8), constrained_layout=True)
plot_curves(["mc"], "critic_rmse", ax=axes[0])
axes[0].set_ylim(0, None)
axes[1].scatter(G_alive[idx], v_alive[idx], s=5, alpha=0.4, color=COLOR["mc"], lw=0)
axes[1].plot([0, 105], [0, 105], color="0.5", ls="--", lw=1)
axes[1].set_xlabel("actual return $G_t$")
axes[1].set_ylabel(r"prediction $V_\phi(s_t)$")
axes[1].set_aspect("equal")
plt.show()

$G_t$ には方策の偶然が残るので、誤差は0にならない。
予測が実際を大きく上回る点（左上）は、500ステップの打ち切り直前の状態である。
状態に残り時間が含まれないので、critic はほかの状態と区別できない。

次は1エピソードに沿った予測である。
500ステップまで立った場合、打ち切りの直前で予測が下がりきらない。

In [ ]:
policy_mc, critic_mc = run["policy"], run["critic"]
torch.manual_seed(1)
states, actions = rollout(lambda s: Categorical(logits=policy_mc(s)).sample())
T = len(actions)
with torch.no_grad():
    v_ep = critic_mc(torch.tensor(states[:T])).squeeze(-1).numpy()  # V_φ(s_0), ..., V_φ(s_{T-1})
G_ep = (1 - 0.99 ** (T - np.arange(T))) / (1 - 0.99)  # 報酬が毎ステップ 1 のときの G_t

fig, ax = plt.subplots(figsize=(5, 2.6))
ax.plot(G_ep, color="0.3", label="actual return $G_t$")
ax.plot(v_ep, color=COLOR["mc"], ls="--", label=r"prediction $V_\phi(s_t)$")
ax.set_xlabel("time step $t$")
ax.set_ylim(0, 110)
ax.legend(frameon=False, fontsize=9)
plt.show()
animate(states, actions, title="value baseline (MC)", values=v_ep)

## 4. TD ターゲット

critic のターゲットを $y_t=r_{t+1}+\gamma V_\phi(s_{t+1})$ に変える。
終了したときは $V_\phi(s_{t+1})=0$ とし、500ステップの打ち切りではそのまま使う。
actor の重み $\gamma^t(G_t-V_\phi(s_t))$ は変えない。

In [ ]:
results["td"] = run_seeds(train_with_critic, critic_target="td")

fig, axes = plt.subplots(1, 2, figsize=(9, 2.8), constrained_layout=True)
plot_curves(["mc", "td"], "return", ax=axes[0])
plot_curves(["mc", "td"], "critic_rmse", ax=axes[1])
axes[1].set_ylim(0, None)
plt.show()

どちらの critic でも方策は学習できる。
TD の critic は、収益が急に伸びる序盤に $G_t$ から大きくずれる（自分の予測の誤差がターゲットに入り、伝わっていく）。
ただし右図は $G_t$ との誤差なので、MC に有利な指標である。

## 5. 発展

`N_SEEDS = 1` にすると早く終わる。

- エピソード数を変える（例：`run_seeds(train_reinforce, M=4)`）
- critic の更新回数を変える（例：`run_seeds(train_with_critic, critic_steps=1)`）
- 重みの $\gamma^t$ を外す（`discount_t` を1にする）
- actor の重みを TD 誤差 $\delta_t=r_{t+1}+\gamma V_\phi(s_{t+1})-V_\phi(s_t)$ にする（Actor–Critic。次回扱う）